<a href="https://colab.research.google.com/github/MavrinIlua/Mavrin-Ilua-Borisovich_ResumeAndPortfolio/blob/main/%D0%9C%D0%B0%D0%B2%D1%80%D0%B8%D0%BD_%D0%98%D0%BB%D1%8C%D1%8F_%D0%91_%D0%90%D1%80%D0%BA%D0%B0%D0%B4%D0%BD%D0%B0%D1%8F_%D0%B8%D0%B3%D1%80%D0%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
html_content = '''
<!DOCTYPE html>
<html>
<head>
    <title>Valentine Catcher Shooter</title>
    <style>
        body {
            margin: 0;
            overflow: hidden;
            display: flex;
            flex-direction: column;
            justify-content: flex-start;
            align-items: center;
            min-height: 100vh;
            background-color: #333;
            font-family: Arial, sans-serif;
        }
        #header {
            width: 800px;
            height: 60px;
            background-color: #222;
            color: white;
            display: flex;
            justify-content: space-between;
            align-items: center;
            padding: 0 20px;
            box-sizing: border-box;
            border-bottom: 2px solid #fff;
        }
        #controls {
            display: flex;
            gap: 10px;
        }
        button {
            padding: 8px 16px;
            font-size: 14px;
            background-color: #444;
            color: white;
            border: 1px solid #666;
            border-radius: 4px;
            cursor: pointer;
        }
        button:hover {
            background-color: #555;
        }
        #gameCanvas {
            background-color: #000;
            border: 2px solid #fff;
            width: 800px;
            height: 540px;
        }
        #help {
            color: #aaa;
            font-size: 14px;
            margin-top: 10px;
        }
        /* СЛОЖНОСТЬ: выбор уровня */
        #difficultyPanel {
            display: flex;
            gap: 10px;
            margin-top: 10px;
        }
    </style>
</head>
<body>

    <!-- ШАПКА ИГРЫ -->
    <div id="header">
        <div>
            Score: <span id="scoreText">0</span> | Lives: <span id="livesText">3</span>
        </div>
        <div id="controls">
            <button id="playBtn">Play</button>
            <button id="pauseBtn">Pause</button>
            <button id="stopBtn">Stop</button>
        </div>
    </div>

    <!-- КАНАВАС -->
    <canvas id="gameCanvas"></canvas>

    <!-- ПОДСКАЗКА -->
    <div id="help">
        W/A/S/D или стрелки: двигаться | Space: стрелять | Q: пауза | R: рестарт | Esc: выход
    </div>

    <!-- ВЫБОР УРОВНЯ СЛОЖНОСТИ -->
    <div id="difficultyPanel">
        <span>Уровень сложности: </span>
        <button id="diffEasy">Easy</button>
        <button id="diffMedium">Medium</button>
        <button id="diffHard">Hard</button>
    </div>

    <script>

        // ПОДГОТОВКА КАНАВАСА
        const canvas = document.getElementById('gameCanvas');
        const ctx = canvas.getContext('2d');
        canvas.width = 800;
        canvas.height = 540;
        const GAME_WIDTH = 800;
        const GAME_HEIGHT = 540;

        // СОСТОЯНИЯ ИГРЫ
        let gameState = 'running';      // 'running' / 'game over' / 'stopped'
        let score = 0;
        let lives = 3;
        let isPaused = false;
        let keys = {};

        // ССЫЛКИ НА UI
        const scoreText = document.getElementById('scoreText');
        const livesText = document.getElementById('livesText');
        const playBtn = document.getElementById('playBtn');
        const pauseBtn = document.getElementById('pauseBtn');
        const stopBtn = document.getElementById('stopBtn');

        const diffEasy = document.getElementById('diffEasy');
        const diffMedium = document.getElementById('diffMedium');
        const diffHard = document.getElementById('diffHard');

        // ПЕРСОНАЖ
        const player = {
            width: 50,
            height: 50,
            x: GAME_WIDTH / 2 - 25,
            y: GAME_HEIGHT - 70,
            speed: 3,                 // базовая скорость: можно менять, чтобы игра была быстрее/медленнее

            direction: 'right',
            canShoot: true,
            shootCooldown: 200,

            draw() {
                const emoji = this.direction === 'left' ? '🧍‍♂️' : '🧍';
                ctx.font = '40px Arial';
                ctx.textAlign = 'center';
                ctx.textBaseline = 'middle';
                ctx.fillText(emoji, this.x + this.width / 2, this.y + this.height / 2);
            }
        };

        // ПУЛИ (СМЕЙЛИКИ-РУКИ / ОБЪЯТИЯ)
        const projectiles = [];
        const projectileEmojis = ['✋', '✊', '🤗', '😽'];

        function Projectile(x, y) {
            this.width = 20;
            this.height = 20;
            this.x = x;
            this.y = y;
            this.speed = 4;          // скорость пульки вверх (меньше = медленнее, виднее смайл)
            this.emoji = projectileEmojis[Math.floor(Math.random() * projectileEmojis.length)];

            this.draw = function() {
                ctx.font = '20px Arial';
                ctx.textAlign = 'center';
                ctx.textBaseline = 'middle';
                ctx.fillText(this.emoji, this.x + this.width / 2, this.y + this.height / 2);
            };
        }

        // ВАЛЕНТИНОК
        const enemies = [];
        const heartEmojis = ['❤️', '💖', '💙', '💚', '💛', '💜'];

        function Enemy(x, y) {
            this.width = 40;
            this.height = 40;
            this.x = x;
            this.y = y;
            this.speed = 1.5;        // базовая скорость падения валентинок
            this.emoji = heartEmojis[Math.floor(Math.random() * heartEmojis.length)];

            this.draw = function() {
                ctx.font = '30px Arial';
                ctx.textAlign = 'center';
                ctx.textBaseline = 'middle';
                ctx.fillText(this.emoji, this.x + this.width / 2, this.y + this.height / 2);
            };
        }

        // ФУНКЦИЯ СОЗДАНИЯ ВАЛЕНТИНОК (интервал будет меняться в зависимости от уровня)
        function generateEnemy() {
            const enemyWidth = 40;
            const randomX = Math.random() * (GAME_WIDTH - enemyWidth);
            const newEnemy = new Enemy(randomX, -enemyWidth);
            enemies.push(newEnemy);
        }

        let enemyGenerationIntervalId;

        // СЛОЖНОСТЬ: параметры для уровней (здесь всё, что можно настроить новичку)
        const difficultyData = {
            easy: {
                enemySpeed: 1.2,          // медленнее падают валентинки
                enemyInterval: 2500,      // реже появляются (в миллисекундах)
                playerLives: 4,           // больше жизней
                canPlayerMove: true,      // можно отключать движение, если хочется (пример)
                description: 'Лёгкий уровень: медленные валентинки и больше жизней.'
            },
            medium: {
                enemySpeed: 1.5,
                enemyInterval: 2000,
                playerLives: 3,
                canPlayerMove: true,
                description: 'Средний уровень: баланс между скоростью и риском.'
            },
            hard: {
                enemySpeed: 2.0,          // быстрее падают
                enemyInterval: 1500,      // чаще появляются
                playerLives: 2,
                canPlayerMove: true,
                description: 'Сложный уровень: быстро и чаще.'
            }
        };

        // Текущий уровень сложности (по умолчанию easy)
        let currentDifficulty = 'easy';

        // ФУНКЦИЯ СБРОСА ИГРЫ (с учётом уровня сложности)
        function resetGame() {
            score = 0;
            // жизни зависят от уровня сложности
            lives = difficultyData[currentDifficulty].playerLives;

            gameState = 'running';
            isPaused = false;

            projectiles.length = 0;
            enemies.length = 0;

            player.x = GAME_WIDTH / 2 - player.width / 2;
            player.y = GAME_HEIGHT - 70;
            player.canShoot = true;
            keys = {};

            // ВАЖНО: меняем скорость валентинок и частоту появления
            Enemy.prototype.speed = difficultyData[currentDifficulty].enemySpeed;
            clearInterval(enemyGenerationIntervalId);
            enemyGenerationIntervalId = setInterval(generateEnemy, difficultyData[currentDifficulty].enemyInterval);

            // Обновляем текст в шапке
            scoreText.textContent = score;
            livesText.textContent = lives;

            console.log('Difficulty:', currentDifficulty, difficultyData[currentDifficulty]);
        }

        function clearCanvas() {
            ctx.clearRect(0, 0, GAME_WIDTH, GAME_HEIGHT);
        }

        // КЛАВИАТУРА
        document.addEventListener('keydown', (e) => {
            keys[e.key] = true;
            e.preventDefault();

            if (gameState === 'stopped') return;

            switch (e.key) {
                case 'q':
                case 'Q':
                    if (gameState === 'running') {
                        isPaused = !isPaused;
                        if (isPaused) {
                            clearInterval(enemyGenerationIntervalId);
                        } else {
                            clearInterval(enemyGenerationIntervalId);
                            enemyGenerationIntervalId = setInterval(
                                generateEnemy, difficultyData[currentDifficulty].enemyInterval
                            );
                        }
                    }
                    break;

                case 'r':
                case 'R':
                    if (gameState === 'game over') {
                        resetGame();
                    }
                    break;

                case 'Escape':
                    if (gameState !== 'stopped') {
                        gameState = 'stopped';
                        isPaused = false;
                        clearInterval(enemyGenerationIntervalId);
                    }
                    break;
            }
        });

        document.addEventListener('keyup', (e) => {
            keys[e.key] = false;
        });

        // КНОПКИ УПРАВЛЕНИЯ
        playBtn.addEventListener('click', () => {
            if (gameState === 'stopped') {
                gameState = 'running';
                isPaused = false;
                enemyGenerationIntervalId = setInterval(
                    generateEnemy, difficultyData[currentDifficulty].enemyInterval
                );
            } else if (gameState === 'game over') {
                resetGame();
            }
        });

        pauseBtn.addEventListener('click', () => {
            if (gameState === 'running') {
                isPaused = !isPaused;
                if (isPaused) {
                    clearInterval(enemyGenerationIntervalId);
                } else {
                    clearInterval(enemyGenerationIntervalId);
                    enemyGenerationIntervalId = setInterval(
                        generateEnemy, difficultyData[currentDifficulty].enemyInterval
                    );
                }
            }
        });

        stopBtn.addEventListener('click', () => {
            if (gameState !== 'stopped') {
                gameState = 'stopped';
                isPaused = false;
                clearInterval(enemyGenerationIntervalId);
            }
        });

        // ВЫБОР УРОВНЯ СЛОЖНОСТИ
        diffEasy.addEventListener('click', () => {
            currentDifficulty = 'easy';
            if (gameState === 'stopped') resetGame();  // если игра остановлена, сразу применяем новый уровень
        });

        diffMedium.addEventListener('click', () => {
            currentDifficulty = 'medium';
            if (gameState === 'stopped') resetGame();
        });

        diffHard.addEventListener('click', () => {
            currentDifficulty = 'hard';
            if (gameState === 'stopped') resetGame();
        });

        // ОБНОВЛЕНИЕ СОСТОЯНИЯ ИГРЫ
        function update() {
            if (gameState !== 'running' || isPaused) return;

            const speed = player.speed;

            player.direction = 'right';

            if (keys['a'] || keys['A'] || keys['ArrowLeft']) {
                player.x = Math.max(0, player.x - speed);
                player.direction = 'left';
            }
            if (keys['d'] || keys['D'] || keys['ArrowRight']) {
                player.x = Math.min(GAME_WIDTH - player.width, player.x + speed);
            }
            if (keys['w'] || keys['W'] || keys['ArrowUp']) {
                player.y = Math.max(0, player.y - speed);
            }
            if (keys['s'] || keys['S'] || keys['ArrowDown']) {
                player.y = Math.min(GAME_HEIGHT - player.height, player.y + speed);
            }

            // Стрельба
            if (keys[' '] && player.canShoot) {
                const newProjectile = new Projectile(player.x + player.width / 2 - 10, player.y);
                projectiles.push(newProjectile);
                player.canShoot = false;
                setTimeout(() => { player.canShoot = true; }, player.shootCooldown);
            }

            // Пули
            for (let i = projectiles.length - 1; i >= 0; i--) {
                projectiles[i].y -= projectiles[i].speed;
                if (projectiles[i].y + projectiles[i].height < 0) {
                    projectiles.splice(i, 1);
                }
            }

            // Валентинки
            for (let i = enemies.length - 1; i >= 0; i--) {
                enemies[i].y += enemies[i].speed;
                if (enemies[i].y > GAME_HEIGHT) {
                    lives--;
                    livesText.textContent = lives;
                    enemies.splice(i, 1);
                    if (lives <= 0) {
                        gameState = 'game over';
                        clearInterval(enemyGenerationIntervalId);
                    }
                }
            }

            // Столкновения
            for (let i = projectiles.length - 1; i >= 0; i--) {
                const projectile = projectiles[i];
                for (let j = enemies.length - 1; j >= 0; j--) {
                    const enemy = enemies[j];
                    if (projectile.x < enemy.x + enemy.width &&
                        projectile.x + projectile.width > enemy.x &&
                        projectile.y < enemy.y + enemy.height &&
                        projectile.y + projectile.height > enemy.y) {

                        score++;
                        scoreText.textContent = score;
                        projectiles.splice(i, 1);
                        enemies.splice(j, 1);
                        break;
                    }
                }
            }
        }

        function drawAll() {
            if (gameState === 'stopped') {
                clearCanvas();
                ctx.fillStyle = 'white';
                ctx.font = '32px Arial';
                ctx.textAlign = 'center';
                ctx.fillText('GAME STOPPED', GAME_WIDTH / 2, GAME_HEIGHT / 2);
                ctx.font = '18px Arial';
                ctx.fillText('Select difficulty and click Play', GAME_WIDTH / 2, GAME_HEIGHT / 2 + 40);
                return;
            }

            clearCanvas();
            player.draw();
            projectiles.forEach(p => p.draw());
            enemies.forEach(e => e.draw());

            if (gameState === 'game over') {
                ctx.fillStyle = 'red';
                ctx.font = '48px Arial';
                ctx.textAlign = 'center';
                ctx.fillText('GAME OVER', GAME_WIDTH / 2, GAME_HEIGHT / 2);
                ctx.font = '24px Arial';
                ctx.fillText('Press R or click Play', GAME_WIDTH / 2, GAME_HEIGHT / 2 + 40);
            }
        }

        function gameLoop() {
            update();
            drawAll();

            if (gameState === 'running' && !isPaused) {
                requestAnimationFrame(gameLoop);
            } else if (gameState === 'stopped' || gameState === 'game over') {
                // игра не обновляется, но цикл остановлен
            }
        }

        // СТАРТОВАЯ НАСТРОЙКА
        resetGame(); // сначала идёт инициализация по уровню "easy"
        playBtn.addEventListener('click', () => { gameLoop(); });
        gameLoop();
    </script>
</body>
</html>
'''

from IPython.display import HTML
HTML(html_content)